<hr>

<h1> Benchmarking </h1>

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import laya
import yaml
import json
import time
import numpy as np

import outlines
from pydantic import BaseModel

c:\Users\Admin\Desktop\ip\Automatic Speech Recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<h3> Mistral 7B</h3>

In [2]:
class LOAD_MODEL:

    """
    Load Model
    """
    def __init__ (self):

        PATH = r"C:\Users\Admin\Desktop\models\Language Models\CPU\Mistral 7B Q4 BnB"

        self.device = "cuda" if torch.cuda.is_available () else "cpu"

        self.TOKENIZER = AutoTokenizer.from_pretrained (PATH)
        self.MODEL = AutoModelForCausalLM.from_pretrained (PATH, device_map = self.device, dtype = torch.float16)

#---------------------#

class EVAL:

    def __init__ (self, MODEL, TOKENIZER, DEVICE):

        self.MODEL = MODEL
        self.TOKENIZER = TOKENIZER
        self.DEVICE = DEVICE

        self.MODEL = outlines.from_transformers (self.MODEL, self.TOKENIZER)

        with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Backend\AssobioChat\SystemPrompts\SYSTEM_PROMPT_SMLAYER.md", "r", encoding = "utf-8") as f:
            self.SYSTEM_PROMPT = f.read ()

        with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Backend\AssobioChat\SystemPrompts\SemanticModel.yaml", "r", encoding = "utf-8") as f:
            self.SEMANTIC_MODEL = yaml.safe_load (f) 

        with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Eval\Semantic-Layer\dataset.json", "r", encoding = "utf-8") as f:
            self.DATASET = json.load (f)

    def STRUCTURED_OUTPUT (self):

        class Format_Constraint (BaseModel):
            operation: str
            columns: str

        PRECISION = []
        LATÊNCIA = []
        for i in range (len(self.DATASET)):

            PROMPT = self.DATASET[i]["prompt"]

            torch.cuda.synchronize ()
            lat = time.time ()

            MENSAGENS = [
                {"role": "system", "content": f"{self.SYSTEM_PROMPT}\n" f"{self.SEMANTIC_MODEL}"},
                {"role": "user", "content": PROMPT}
            ]

            MENSAGENS = self.TOKENIZER.apply_chat_template (MENSAGENS, tokenize = False, add_generation_prompt = True)
            #print (MENSAGENS)

            SEMANTIC_QUERY = self.MODEL (MENSAGENS, output_type = Format_Constraint, max_new_tokens = 50)
            SEMANTIC_QUERY = json.loads (SEMANTIC_QUERY)

            torch.cuda.synchronize ()
            lat = time.time () - lat
            LATÊNCIA.append (lat)
            ##Eval

            if SEMANTIC_QUERY["operation"] == self.DATASET[i]["expected_output"]["operation"] and SEMANTIC_QUERY["columns"] == self.DATASET[i]["expected_output"]["columns"]:
                PRECISION.append (1)

            else:
                PRECISION.append (0)

        return PRECISION, LATÊNCIA


if __name__ == "__main__":

    MODELO = LOAD_MODEL ()
    #print (dir(MODELO))
    BENCH = EVAL (MODELO.MODEL, MODELO.TOKENIZER, MODELO.device)
    PRECISION, LATÊNCIA = BENCH.STRUCTURED_OUTPUT ()

    print (PRECISION)
    print (LATÊNCIA)


W0922 12:19:12.151000 7436 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 291/291 [00:04<00:00, 64.62it/s]
[transformers] Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
W0922 12:19:22.957000 7436 Lib\site-packages\torch\_dynamo\convert_frame.py:2415] WON'T CONVERT _apply_token_bitmask_inplace_kernel c:\Users\Admin\Desktop\ip\Automatic Speech Recognition\.venv\Lib\site-packages\outlines_core\kernels\torch.py line 43 
W0922 12:19:22.957000 7436 Lib\site-packages\torch\_dynamo\convert_frame.py:2415] due to: 
W0922 12:19:22.957000 7436 Lib\site-packages\torch\_dynamo\convert_frame.py:2415] Traceback (most recent call last):
W0922 12:19:22.957000 7436 Lib\site-packages\torch\_dynamo\convert

[1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[4.8201282024383545, 3.931861162185669, 3.0665721893310547, 2.605168342590332, 2.894778251647949, 2.5998952388763428, 2.386235237121582, 2.912395715713501, 2.6012678146362305, 3.133300304412842, 2.4498050212860107, 2.466714859008789, 2.9358580112457275, 2.447601556777954, 2.7830584049224854, 3.066803455352783, 2.149873733520508, 2.450026273727417, 2.233243227005005, 2.6161186695098877, 2.9336304664611816, 2.6168978214263916, 3.933413028717041, 2.4664599895477295, 2.3668651580810547, 2.46657657623291, 3.933274507522583, 2.2327277660369873, 3.083951234817505, 2.020966053009033, 3.012636661529541, 2.6162664890289307, 2.466843843460083, 2.183077573776245, 3.0166702270507812, 2.403350830078125, 2.929947853088379, 2.9384634494781494, 2.8446156978607178, 2.2505669593811035, 3.1495935916900635, 2.0833151340

In [3]:
print (PRECISION)
print (f"Média Precisão: {np.mean (PRECISION)}")

print ("---" *50)

print (LATÊNCIA)
print (f"Média Latência: {np.mean (LATÊNCIA)}")
print (f"P90 Latência: {np.percentile (LATÊNCIA, 95)}")

[1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Média Precisão: 0.8571428571428571
------------------------------------------------------------------------------------------------------------------------------------------------------
[4.8201282024383545, 3.931861162185669, 3.0665721893310547, 2.605168342590332, 2.894778251647949, 2.5998952388763428, 2.386235237121582, 2.912395715713501, 2.6012678146362305, 3.133300304412842, 2.4498050212860107, 2.466714859008789, 2.9358580112457275, 2.447601556777954, 2.7830584049224854, 3.066803455352783, 2.149873733520508, 2.450026273727417, 2.233243227005005, 2.6161186695098877, 2.9336304664611816, 2.6168978214263916, 3.933413028717041, 2.4664599895477295, 2.3668651580810547, 2.46657657623291, 3.933274507522583, 2.2327277660369873, 3.083951234817505, 2.020966053009033, 3.012636661529541, 2.6162664890289307, 2.

<h3> Laya em Português Europeu </h3>

In [2]:
class LOAD_MODEL:

    def __init__ (self):

        self.LAYA = laya.load (r"C:\Users\Admin\Desktop\models\System One Models\Laya", device = "cuda")

        #----------#


class LAYA_EVAL:

    def __init__ (self, laya):

        self.Laya_Model = laya

        with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Eval\Semantic-Layer\dataset.json", "r", encoding = "utf-8") as f:
            self.DATASET = json.load (f)


    def Laya_Eval (self):

        PRECISION = []
        LATÊNCIA = []

        for i in range (len(self.DATASET)):

            STATE = self.DATASET[i]["prompt"]

            QUESTIONS = {
                "OPERATION": {
                    "type": "choice",
                    "instructions": (
                        "Seleciona a operação que deve ser aplicada à base de dados "
                        "para responder ao pedido do utilizador. Escolhe apenas uma "
                        "operação."
                    ),
                    "criteria": {
                        "RETURN": (
                            "Retornar os valores existentes da coluna solicitada, "
                            "sem realizar qualquer agregação ou cálculo."
                        ),
                        "COUNT": (
                            "Contar o número de registos da coluna solicitada."
                        ),
                        "MEAN": (
                            "Calcular a média aritmética dos valores numéricos "
                            "da coluna solicitada."
                        ),
                        "SUM": (
                            "Calcular a soma de todos os valores numéricos "
                            "da coluna solicitada."
                        ),
                    }
                },

                "COLUMN": {
                    "type": "choice",
                    "instructions": (
                        "Seleciona a coluna da base de dados que contém os dados "
                        "necessários para responder ao pedido do utilizador. "
                        "Escolhe apenas uma coluna."
                    ),
                    "criteria": {
                        "AUDIO_TIME": (
                            "Duração do áudio original, em segundos."
                        ),

                        "TEMPO_PRÉ_PROCESSAMENTO": (
                            "Tempo necessário para realizar o pré-processamento "
                            "do áudio, em segundos."
                        ),

                        "TRANSCRIÇÃO": (
                            "Texto resultante da transcrição produzida pelo "
                            "modelo de Automatic Speech Recognition (ASR)."
                        ),

                        "TEMPO_PROCESSAMENTO_MODELO_ASR": (
                            "Tempo total de processamento do modelo ASR, "
                            "em segundos."
                        ),

                        "TEMPO_INFERÊNCIA_MODELO_ASR": (
                            "Tempo exclusivamente gasto na inferência do modelo "
                            "ASR, em segundos."
                        ),

                        "LATÊNCIA": (
                            "Latência total do pipeline ASR desde o início do "
                            "processamento até à disponibilização da transcrição, "
                            "em segundos."
                        ),

                        "TOKENS_PER_SECOND_DECODE": (
                            "Throughput do processo de decode do modelo ASR, "
                            "medido em tokens por segundo."
                        ),

                        "HARDWARE_LLM": (
                            "Hardware utilizado para executar o Large Language Model."
                        ),

                        "MODELO_LLM": (
                            "Large Language Model utilizado para realizar a auditoria."
                        ),

                        "AUDITORIA_LLM": (
                            "Resultado ou conteúdo da auditoria realizada pelo "
                            "Large Language Model."
                        ),

                        "NÚMERO_DE_TOKENS_PROCESSADOS": (
                            "Número de tokens processados pelo Large Language Model."
                        ),

                        "TEMPO_PREFILL_LLM": (
                            "Tempo gasto na fase de prefill do Large Language Model, "
                            "em segundos."
                        ),

                        "TOKENS_PER_SECOND_PREFILL_LLM": (
                            "Throughput da fase de prefill do Large Language Model, "
                            "em tokens por segundo."
                        ),

                        "TEMPO_DECODE_LLM": (
                            "Tempo gasto na fase de decode do Large Language Model, "
                            "em segundos."
                        ),

                        "TOKENS_PER_SECOND_DECODE_LLM": (
                            "Throughput da fase de decode do Large Language Model, "
                            "em tokens por segundo."
                        ),

                        "LATÊNCIA_LLM": (
                            "Tempo total necessário para o Large Language Model "
                            "realizar a auditoria, em segundos."
                        ),
                    }
                }
            }

            torch.cuda.synchronize ()
            lat = time.time ()

            OUTPUT = self.Laya_Model.predict (STATE, QUESTIONS)

            torch.cuda.synchronize ()
            lat = time.time () - lat 
            LATÊNCIA.append (lat)

            if OUTPUT["answers"]["OPERATION"]["choice"] == self.DATASET[i]["expected_output"]["operation"] and OUTPUT["answers"]["COLUMN"]["choice"] == self.DATASET[i]["expected_output"]["columns"]:

                PRECISION.append (1)

            else:

                PRECISION.append (0)


        return PRECISION, LATÊNCIA


if __name__ == "__main__":

    MODEL = LOAD_MODEL ()
    #print (dir(MODEL))
    BENCH = LAYA_EVAL (MODEL.LAYA)
    PRECISION, LATÊNCIA = BENCH.Laya_Eval ()

    print (PRECISION)
    print (LATÊNCIA)


[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0]
[0.4508206844329834, 0.04825615882873535, 0.03352499008178711, 0.03490138053894043, 0.033652305603027344, 0.03177189826965332, 0.033516883850097656, 0.03517889976501465, 0.03175950050354004, 0.03199958801269531, 0.03446340560913086, 0.0324552059173584, 0.03202342987060547, 0.039797067642211914, 0.03182673454284668, 0.030519962310791016, 0.03766918182373047, 0.04414772987365723, 0.03238177299499512, 0.02361154556274414, 0.047888755798339844, 0.03160405158996582, 0.037008047103881836, 0.029063940048217773, 0.03324556350708008, 0.03783750534057617, 0.0317234992980957, 0.03155398368835449, 0.032036781311035156, 0.03735756874084473, 0.0448613166809082, 0.034682273864746094, 0.020237445831298828, 0.04644322395324707, 0.037682533264160156, 0.028249025344848633, 0.032097816467285156, 0.03172492980957031, 0.

In [3]:
print (PRECISION)
print (len(PRECISION))
print (f"Média Precisão: {np.mean (PRECISION)}")

print ("---" *50)

print (LATÊNCIA)
print (f"Média Latência: {np.mean (LATÊNCIA)}")
print (f"P90 Latência: {np.percentile (LATÊNCIA, 95)}")

[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0]
63
Média Precisão: 0.5079365079365079
------------------------------------------------------------------------------------------------------------------------------------------------------
[0.4508206844329834, 0.04825615882873535, 0.03352499008178711, 0.03490138053894043, 0.033652305603027344, 0.03177189826965332, 0.033516883850097656, 0.03517889976501465, 0.03175950050354004, 0.03199958801269531, 0.03446340560913086, 0.0324552059173584, 0.03202342987060547, 0.039797067642211914, 0.03182673454284668, 0.030519962310791016, 0.03766918182373047, 0.04414772987365723, 0.03238177299499512, 0.02361154556274414, 0.047888755798339844, 0.03160405158996582, 0.037008047103881836, 0.029063940048217773, 0.03324556350708008, 0.03783750534057617, 0.0317234992980957, 0.03155398368835449, 0.032036781311035156, 0.0373

<h3> Laya em Inglês </h3>

In [2]:
class LOAD_MODEL:

    def __init__ (self):

        self.LAYA = laya.load (r"C:\Users\Admin\Desktop\models\System One Models\Laya", device = "cuda")

        #----------#


class LAYA_EVAL:

    def __init__ (self, laya):

        self.Laya_Model = laya

        with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Eval\Semantic-Layer\dataset.json", "r", encoding = "utf-8") as f:
            self.DATASET = json.load (f)


    def Laya_Eval (self):

        ACCURACY = []
        LATÊNCIA = []

        for i in range (len(self.DATASET)):

            STATE = self.DATASET[i]["prompt_eng"]

            
            QUESTIONS = {
                "OPERATION": {
                    "type": "choice",
                    "instructions": (
                        "Select the operation that should be applied to the database "
                        "to answer the user's request. Select only one operation."
                    ),
                    "criteria": {
                        "RETURN": (
                            "Return the existing values from the requested column "
                            "without performing any aggregation or calculation."
                        ),
                        "COUNT": (
                            "Count the number of records in the requested column."
                        ),
                        "MEAN": (
                            "Calculate the arithmetic mean of the numeric values "
                            "in the requested column."
                        ),
                        "SUM": (
                            "Calculate the sum of all numeric values "
                            "in the requested column."
                        ),
                    }
                },

                "COLUMN": {
                    "type": "choice",
                    "instructions": (
                        "Select the database column that contains the data "
                        "required to answer the user's request. Select only one column."
                    ),
                    "criteria": {
                        "AUDIO_TIME": (
                            "Duration of the original audio, in seconds."
                        ),
                        "TEMPO_PRÉ_PROCESSAMENTO": (
                            "Time required to perform audio preprocessing, in seconds."
                        ),
                        "TRANSCRIÇÃO": (
                            "Text resulting from the transcription produced by "
                            "the Automatic Speech Recognition (ASR) model."
                        ),
                        "TEMPO_PROCESSAMENTO_MODELO_ASR": (
                            "Total processing time of the ASR model, in seconds."
                        ),
                        "TEMPO_INFERÊNCIA_MODELO_ASR": (
                            "Time exclusively spent performing inference "
                            "with the ASR model, in seconds."
                        ),
                        "LATÊNCIA": (
                            "Total latency of the ASR pipeline, from the beginning "
                            "of processing until the transcription is available, "
                            "in seconds."
                        ),
                        "TOKENS_PER_SECOND_DECODE": (
                            "Throughput of the ASR model's decoding process, "
                            "measured in tokens per second."
                        ),
                        "HARDWARE_LLM": (
                            "Hardware used to run the Large Language Model."
                        ),
                        "MODELO_LLM": (
                            "Large Language Model used to perform the audit."
                        ),
                        "AUDITORIA_LLM": (
                            "Result or content of the audit performed "
                            "by the Large Language Model."
                        ),
                        "NÚMERO_DE_TOKENS_PROCESSADOS": (
                            "Number of tokens processed by the Large Language Model."
                        ),
                        "TEMPO_PREFILL_LLM": (
                            "Time spent during the prefill phase of the "
                            "Large Language Model, in seconds."
                        ),
                        "TOKENS_PER_SECOND_PREFILL_LLM": (
                            "Throughput of the prefill phase of the "
                            "Large Language Model, in tokens per second."
                        ),
                        "TEMPO_DECODE_LLM": (
                            "Time spent during the decode phase of the "
                            "Large Language Model, in seconds."
                        ),
                        "TOKENS_PER_SECOND_DECODE_LLM": (
                            "Throughput of the decode phase of the "
                            "Large Language Model, in tokens per second."
                        ),
                        "LATÊNCIA_LLM": (
                            "Total time required by the Large Language Model "
                            "to perform the audit, in seconds."
                        ),
                    }
                }
            }



            torch.cuda.synchronize ()
            lat = time.time ()

            OUTPUT = self.Laya_Model.predict (STATE, QUESTIONS)

            torch.cuda.synchronize ()
            lat = time.time () - lat 
            LATÊNCIA.append (lat)

            if OUTPUT["answers"]["OPERATION"]["choice"] == self.DATASET[i]["expected_output"]["operation"] and OUTPUT["answers"]["COLUMN"]["choice"] == self.DATASET[i]["expected_output"]["columns"]:

                ACCURACY.append (1)

            else:

                ACCURACY.append (0)


        return ACCURACY, LATÊNCIA


if __name__ == "__main__":

    MODEL = LOAD_MODEL ()
    #print (dir(MODEL))
    BENCH = LAYA_EVAL (MODEL.LAYA)
    ACCURACY, LATÊNCIA = BENCH.Laya_Eval ()

    print (ACCURACY)
    print (LATÊNCIA)


[1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1]
[0.45049619674682617, 0.04383063316345215, 0.03545975685119629, 0.032201290130615234, 0.03246283531188965, 0.03230714797973633, 0.035294294357299805, 0.032362937927246094, 0.03362131118774414, 0.03404664993286133, 0.03175044059753418, 0.0317997932434082, 0.0413050651550293, 0.027828454971313477, 0.03386974334716797, 0.03818011283874512, 0.04413247108459473, 0.03437089920043945, 0.033503055572509766, 0.03223896026611328, 0.031877756118774414, 0.03869009017944336, 0.03385019302368164, 0.029909133911132812, 0.03813886642456055, 0.028565406799316406, 0.03319191932678223, 0.03835439682006836, 0.027720928192138672, 0.033272743225097656, 0.033297061920166016, 0.03426480293273926, 0.032363176345825195, 0.03419804573059082, 0.04944586753845215, 0.03381848335266113, 0.03541731834411621, 0.03055715560913086, 0

In [3]:
print (ACCURACY)
print (len(ACCURACY))
print (f"Média Precisão: {np.mean (ACCURACY)}")

print ("---" *50)

print (LATÊNCIA)
print (f"Média Latência: {np.mean (LATÊNCIA)}")
print (f"P90 Latência: {np.percentile (LATÊNCIA, 95)}")

[1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1]
63
Média Precisão: 0.6507936507936508
------------------------------------------------------------------------------------------------------------------------------------------------------
[0.45049619674682617, 0.04383063316345215, 0.03545975685119629, 0.032201290130615234, 0.03246283531188965, 0.03230714797973633, 0.035294294357299805, 0.032362937927246094, 0.03362131118774414, 0.03404664993286133, 0.03175044059753418, 0.0317997932434082, 0.0413050651550293, 0.027828454971313477, 0.03386974334716797, 0.03818011283874512, 0.04413247108459473, 0.03437089920043945, 0.033503055572509766, 0.03223896026611328, 0.031877756118774414, 0.03869009017944336, 0.03385019302368164, 0.029909133911132812, 0.03813886642456055, 0.028565406799316406, 0.03319191932678223, 0.03835439682006836, 0.027720928192138672, 0.03